# NB_lab_test_trends
Builds **`<catalog>.gold.lab_test_trends`** from the silver layer, per STTM *Silver to Gold (CareSync Health Network) v1.0*.

* **Grain:** one row per test type (`test_name`) per calendar month
* **Driving table:** `<catalog>.silver.lab_results` grouped by `test_name`, `year(test_date)`, `month(test_date)`
* **Joins:** none required; optional LEFT JOIN `silver.calendar` on `date_key = date_format(date_trunc('month', test_date), 'yyyyMMdd')` (cast to INT)
* **Window:** `LAG(abnormal_rate_pct)` partitioned by `test_name`, ordered by `year`, `month` (month-over-month trend)
* **Filters:** none specified
* **Load type:** FULL, `CREATE OR REPLACE TABLE ... AS SELECT`

Note: `silver.calendar` has no `quarter_name` column, so `quarter_name` is derived as `'Q' || calendar.quarter` (e.g. "Q1").

In [0]:
import re

dbutils.widgets.text("catalog", "adb_caresync_apollo", "Catalog")
catalog = dbutils.widgets.get("catalog").strip()

# Guard against injection: catalog must be a plain identifier
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog):
    raise ValueError(f"Invalid catalog name: {catalog!r}")

silver = f"`{catalog}`.silver"
gold = f"`{catalog}`.gold"
target_table = f"{gold}.lab_test_trends"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold}")
print(f"Target: {target_table}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per test type per month with abnormal/critical rates and month-over-month trend. Source: STTM silver_to_gold v1.0'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                       AS year,
    month(test_date)                      AS month,
    MIN(date_trunc('MONTH', test_date))   AS month_start,
    COUNT(*)                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count
  FROM {silver}.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    m.*,
    CAST(abnormal_test_count / total_tests * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(critical_test_count / total_tests * 100 AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT
    r.*,
    LAG(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  CAST(w.test_name AS STRING)            AS test_name,
  CAST(w.year AS INT)                    AS year,
  CAST(w.month AS INT)                   AS month,
  CAST(c.month_name AS STRING)           AS month_name,
  CAST(CONCAT('Q', c.quarter) AS STRING) AS quarter_name,
  CAST(w.total_tests AS INT)             AS total_tests,
  CAST(w.abnormal_test_count AS INT)     AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)     AS critical_test_count,
  w.abnormal_rate_pct                    AS abnormal_rate_pct,
  w.critical_rate_pct                    AS critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2)) AS prev_month_abnormal_rate_pct,
  CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN {silver}.calendar c
  ON c.date_key = CAST(date_format(w.month_start, 'yyyyMMdd') AS INT)
""")

In [0]:
updated_df = spark.table(target_table)
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))